# Week 8 (Tue) - Class Activities in CS 315 (DuckDB Version)

Solving tasks from `netflix_data.db` using DuckDB SQL queries.

**Table of Contents**

0. [Q0: Connect to Database & List Tables](#q0)
1. [Q1: Inspect Table Data](#q1)
2. [Q2: Show Number of Rows](#q2)
3. [Q3: Show Column Names](#q3)
4. [Q4: Show Number of Total Countries](#q4)
5. [Q5: Show Number of TV Series](#q5)
6. [Q6: Number of TV Series in South Korea](#q6)
7. [Q7: Total Weeks of Squid Game in Italy](#q7)
8. [Q8: Time Frame for a Given Country](#q8)
9. [Q9: Is Number of Weeks in Ranking Equal?](#q9)
10. [Q10: Country with Fewest Weeks](#q10)

<a id="q0"></a>
### Q0: Connect to Database & List Tables

Connect to `netflix_data.db` and inspect the available tables in the database.

In [4]:
%pip install duckdb


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import duckdb

con = duckdb.connect('netflix_data.db')
con

When we open a connection it will remain open until we explicetely close it.

We can execute simple SQL commands as shown below.

In [2]:
con.sql("SHOW TABLES")

nameVARCHAR
films
tvseries


<a id="q1"></a>
### Q1: Load & Preview Data

Preview the top rows `tvseries`. This is similar to the `.head` method in pandas.

In [3]:
con.sql("SELECT * FROM tvseries LIMIT 5")

country_nameVARCHAR,country_iso2VARCHAR,weekVARCHAR,weekly_rankBIGINT,show_titleVARCHAR,season_titleVARCHAR,cumulative_weeks_in_top_10BIGINT
Argentina,AR,2026-10-04,1,Not a Stranger,Not a Stranger: Season 1,3
Argentina,AR,2026-10-04,2,My Sad Dead,My Sad Dead: Limited Series,2
Argentina,AR,2026-10-04,3,The Final Problem,The Final Problem: Limited Series,2
Argentina,AR,2026-10-04,4,Shaque: Trust No One,Shaque: Trust No One: Season 1,1
Argentina,AR,2026-10-04,5,East of Eden,East of Eden: Limited Series,1


<a id="q2"></a>
### Q2: Show number of rows

In [4]:
con.sql("SELECT COUNT(*) FROM tvseries")

count_star()BIGINT
256100


Because the result is a table, if we want just the number, we will use the method `fetchone` to get the first row:

In [9]:
con.execute("SELECT COUNT(*) FROM tvseries").fetchone()

(256100,)

Since the result is a tuple, then we can index it like this:

In [11]:
con.execute("SELECT COUNT(*) FROM tvseries").fetchone()[0]

256100

<a id="q3"></a>
### Q3: Show column names

In [14]:
con.execute("DESCRIBE tvseries")

<a id="q4"></a>
### Q4: Show number of total countries

In [16]:
con.execute("SELECT COUNT(DISTINCT country_name) FROM tvseries").fetchone()[0]

94

<a id="q5"></a>
### Q5: Show number of TV series

In [18]:
con.execute("SELECT COUNT(DISTINCT show_title) FROM tvseries").fetchone()[0]

3574

<a id="q6"></a>
### Q6: Number of TV series in South Korea

In [20]:
con.execute("""
    SELECT COUNT(DISTINCT show_title) 
    FROM tvseries 
    WHERE country_name = 'South Korea'
""").fetchone()[0]

481

<a id="q7"></a>
### Q7: Total weeks of Squid Game in Italy

In [24]:
con.execute("""
    SELECT COUNT(DISTINCT week) 
    FROM tvseries 
    WHERE country_name = 'Italy' AND show_title = 'Squid Game'
""").fetchone()[0]

29

<a id="q8"></a>
### Q8: Time frame for a given country

Write a function that given a country name prints out the start and end date of their rankings.

Once you have created the function, call it for a few countries to test it.

In [28]:
def show_timeframe(con, country):
    start, end = con.execute("""
        SELECT MIN(week), MAX(week)
        FROM tvseries
        WHERE country_name = $1
    """, [country]).fetchone()

    print(f"Netflix rankings for {country} exist between {start} and {end}.")

show_timeframe(con, "Chile")
show_timeframe(con, "Russia")

Netflix rankings for Chile exist between 2021-07-04 and 2026-10-04.
Netflix rankings for Russia exist between 2021-07-04 and 2022-02-27.


<a id="q9"></a>
### Q9: Is number of weeks equal for each country?

Check if total rows match `20 * unique_weeks * unique_countries`.

In [32]:
is_equal = con.execute("""
    SELECT COUNT(*) = 10 * COUNT(DISTINCT week) * COUNT(DISTINCT country_name)
    FROM tvseries
""").fetchone()[0]

is_equal

False

We should have expected this to be `False` since we saw in Q8 that Russia stopped being in the rankings after it attacked Ukraine in February 2022.

<a id="q10"></a>
### Q10: Country with fewest weeks

Find the country with the fewest weeks in the Netflix rankings.

In [35]:
con.abort()

In [38]:
con.execute("SELECT COUNT(*) FROM tvseries").fetchall()[0][0]
def show_timeframe(con, country):
    start, end = con.execute("""
        SELECT MIN(week), MAX(week)
        FROM tvseries
        WHERE country_name = $1
    """, [country]).fetchall()[0]

    print(f"Netflix rankings for {country} exist between {start} and {end}.")
is_equal = con.execute("""
    SELECT COUNT(*) = 10 * COUNT(DISTINCT week) * COUNT(DISTINCT country_name)
    FROM tvseries""").fetchall()[0][0]

is_equal


False